# Semana 7: Práctica. La frontera eficiente con datos reales

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/07_markowitz_frontera/clase07_practica.ipynb)

La semana pasada medimos el universo; hoy lo optimizamos. Con las mismas 14 acciones construimos la nube de portafolios posibles, la frontera eficiente, el portafolio de mínima varianza, el tangente y la CAL. Después hacemos el experimento incómodo: ver cuánto cambian los pesos "óptimos" cuando cambian los datos. La última parte es el reforzamiento rumbo al examen parcial.

**Requisito previo:** `git pull` en tu fork para tener `utils/finanzas.py` actualizado (esta semana se agregan `min_varianza()`, `portafolio_tangente()` y `frontera_eficiente()`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import (retornos, riesgo_portafolio, min_varianza,
                            portafolio_tangente, frontera_eficiente)

UNIVERSO = ["ALICORC1.LM", "FERREYC1.LM", "CPACASC1.LM", "LUSURC1.LM",      # BVL (en soles)
            "BAP", "BVN", "SCCO", "IFS",                                    # Peru en NYSE
            "AAPL", "MSFT", "JNJ", "KO", "XOM", "JPM"]                      # Estados Unidos

## 1. Los insumos: $\mu$, $\Sigma$ y $R_f$

Mismo tratamiento de la semana 6: 5 años de precios mensuales ajustados, todo convertido a dólares y sin el mes en curso. De ahí salen los dos insumos de Markowitz, anualizados.

Falta el tercero: la tasa libre de riesgo. **Regla de consistencia de la semana 2:** misma moneda y mismo horizonte que los retornos. Nuestros retornos están en dólares, así que usamos la letra del Tesoro de Estados Unidos a 3 meses (`^IRX` en Yahoo, en porcentaje anual), no una tasa en soles.

In [ ]:
bruto = yf.download(UNIVERSO + ["PEN=X"], period="5y", interval="1mo",
                    auto_adjust=True, progress=False)["Close"].iloc[:-1]
tc = bruto["PEN=X"].ffill()
precios = bruto.drop(columns="PEN=X")
locales = [t for t in precios.columns if t.endswith(".LM")]
precios[locales] = precios[locales].div(tc, axis=0)                     # de soles a dolares
precios = precios.dropna(axis=1, thresh=48).ffill()
UNIVERSO = [t for t in UNIVERSO if t in precios.columns]

ret = retornos(precios[UNIVERSO])
mu = ret.mean() * 12                                                    # retornos esperados anualizados
cov = ret.cov() * 12                                                    # matriz de covarianzas anualizada

try:
    rf = float(yf.download("^IRX", period="1mo", progress=False)["Close"].dropna().iloc[-1].item()) / 100
except Exception:
    rf = 0.04                                                           # respaldo si Yahoo no responde
print(f"{len(UNIVERSO)} acciones, {len(ret)} meses | rf (T-bill 3 meses, USD) = {rf:.2%}")
pd.DataFrame({"mu": mu, "sigma": np.sqrt(np.diag(cov)), "Sharpe": (mu - rf) / np.sqrt(np.diag(cov))}).round(3).sort_values("Sharpe")

**Una advertencia antes de optimizar:** mira los retornos medios de la tabla. Varios superan el 25 o 30 por ciento anual. ¿Alguien cree que esos son los retornos *esperados* de los próximos años? Son el promedio de una muestra de 5 años que resultó buena para esos activos. Vamos a usarlos porque es lo que hace el método en su versión de libro de texto, y en la sección 4 veremos lo que cuesta.

## 2. La nube Monte Carlo: todas las mezclas que podamos imaginar

Antes de optimizar, fuerza bruta: 20 000 portafolios con pesos al azar (positivos y que suman 1). Cada punto es un portafolio. ¿Se ve un borde?

In [ ]:
rng = np.random.default_rng(421)
N = 20_000
# Mitad con pesos dispersos y mitad concentrados, para poblar tambien los bordes de la nube
W = np.vstack([rng.dirichlet(np.ones(len(UNIVERSO)), N // 2),
               rng.dirichlet(np.full(len(UNIVERSO), 0.2), N // 2)])

mc_ret = W @ mu.values
mc_vol = np.sqrt(np.einsum("ij,jk,ik->i", W, cov.values, W))
mc_sharpe = (mc_ret - rf) / mc_vol

mejor = mc_sharpe.argmax()
print(f"Mejor Sharpe de la nube: {mc_sharpe[mejor]:.3f} (E = {mc_ret[mejor]:.1%}, sigma = {mc_vol[mejor]:.1%})")
print(f"Menor riesgo de la nube: {mc_vol.min():.1%}")

## 3. La frontera eficiente, el PMV, el tangente y la CAL

Ahora con optimizador. Tres líneas de la librería hacen lo que la nube solo aproxima. Todo sin ventas en corto (`cortos=False` es el valor por defecto).

In [ ]:
w_mv = min_varianza(cov)
w_t = portafolio_tangente(mu, cov, rf)
fe = frontera_eficiente(mu, cov, n_puntos=40)

def medir(w):
    e, s = float(w @ mu), riesgo_portafolio(w, cov)
    return e, s, (e - rf) / s

(e_mv, s_mv, sh_mv), (e_t, s_t, sh_t) = medir(w_mv), medir(w_t)
print(f"PMV:      E = {e_mv:.1%}  sigma = {s_mv:.1%}  Sharpe = {sh_mv:.3f}")
print(f"Tangente: E = {e_t:.1%}  sigma = {s_t:.1%}  Sharpe = {sh_t:.3f}   (la nube habia llegado a {mc_sharpe[mejor]:.3f})")

fig, ax = plt.subplots(figsize=(9.5, 6))
nube = ax.scatter(mc_vol * 100, mc_ret * 100, c=mc_sharpe, cmap="viridis", s=3, alpha=0.5)
fig.colorbar(nube, label="Ratio de Sharpe")
ax.plot(fe["riesgo"] * 100, fe["retorno"] * 100, color="#000080", lw=2.5, label="Frontera eficiente")
x_cal = np.array([0, fe["riesgo"].max()])
ax.plot(x_cal * 100, (rf + sh_t * x_cal) * 100, color="#E36C0A", lw=2, label="CAL optima")
ax.scatter(np.sqrt(np.diag(cov)) * 100, mu * 100, color="black", s=25, zorder=3)
for t in UNIVERSO:
    ax.annotate(t.replace(".LM", ""), (np.sqrt(cov.loc[t, t]) * 100, mu[t] * 100),
                xytext=(4, 3), textcoords="offset points", fontsize=7)
ax.scatter(s_mv * 100, e_mv * 100, marker="D", color="#006E6E", s=90, zorder=4, label="Minima varianza (PMV)")
ax.scatter(s_t * 100, e_t * 100, marker="*", color="#E36C0A", s=260, zorder=4, label="Tangente")
ax.scatter(0, rf * 100, color="black", zorder=4); ax.annotate("rf", (0, rf * 100), xytext=(5, 5), textcoords="offset points")
ax.set_xlim(left=-1); ax.set_ylim(top=mu.max() * 110); ax.set_xlabel("Volatilidad anualizada (%)"); ax.set_ylabel("Retorno medio anualizado (%)")
ax.set_title("Nube Monte Carlo, frontera eficiente y CAL"); ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Preguntas de discusión:** ¿la nube llega hasta la frontera? (Casi nunca: los pesos al azar rara vez dan portafolios concentrados, y la frontera vive en los bordes. Por eso se optimiza en lugar de simular.) ¿Hay alguna acción individual sobre la frontera? ¿Por qué la CAL queda por encima de toda la frontera salvo en un punto?

Ahora lo más importante: **qué hay dentro** del PMV y del tangente.

In [ ]:
pesos = pd.DataFrame({"PMV": w_mv, "Tangente": w_t, "Pesos iguales": 1 / len(UNIVERSO)})
pesos.index = [t.replace(".LM", "") for t in pesos.index]

ax = pesos[["PMV", "Tangente"]].mul(100).plot.bar(figsize=(9.5, 4), color=["#006E6E", "#E36C0A"], width=0.8)
ax.axhline(100 / len(UNIVERSO), color="gray", ls=":", label="Pesos iguales")
ax.set_ylabel("Peso (%)"); ax.set_title("Que hay dentro de cada portafolio"); ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print(f"Activos con peso mayor a 1%: PMV = {(w_mv > 0.01).sum()} | Tangente = {(w_t > 0.01).sum()} (de {len(UNIVERSO)})")
print(f"Mayor peso individual: PMV = {w_mv.max():.0%} | Tangente = {w_t.max():.0%}")

**Para discutir:** ¿cuántos activos usa realmente el tangente? ¿Le presentarías ese portafolio a un comité de inversiones? Fíjate en quiénes son los favoritos: los que tuvieron el mejor Sharpe **en esta muestra**. El optimizador no distingue habilidad de suerte.

## 4. El experimento incómodo: ¿qué tan estables son los pesos?

Nuestros 58 meses son una sola muestra de las muchas que pudieron ocurrir. Simulamos otras historias posibles con un **bootstrap**: sorteamos meses con reemplazo hasta armar una muestra del mismo tamaño, reestimamos $\mu$ y $\Sigma$, y optimizamos de nuevo. Lo repetimos 300 veces. Si el método fuera robusto, los pesos deberían moverse poco.

In [ ]:
rng = np.random.default_rng(7)
boot_t, boot_mv = [], []
for _ in range(300):
    r = ret.iloc[rng.integers(0, len(ret), len(ret))]           # meses sorteados con reemplazo
    m, c = r.mean() * 12, r.cov() * 12
    boot_t.append(portafolio_tangente(m, c, rf).values)
    boot_mv.append(min_varianza(c).values)
boot_t, boot_mv = np.array(boot_t), np.array(boot_mv)

etiquetas = [t.replace(".LM", "") for t in UNIVERSO]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, datos, w_muestra, titulo in [(axes[0], boot_t, w_t, "Tangente: usa mu y la covarianza"),
                                     (axes[1], boot_mv, w_mv, "PMV: solo usa la covarianza")]:
    ax.boxplot(datos * 100, showfliers=False)
    ax.set_xticks(range(1, len(UNIVERSO) + 1)); ax.set_xticklabels(etiquetas, rotation=90)
    ax.scatter(range(1, len(UNIVERSO) + 1), w_muestra.values * 100, color="#E36C0A", zorder=3, label="Peso con la muestra original")
    ax.set_title(titulo); ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
axes[0].set_ylabel("Peso (%)"); plt.tight_layout(); plt.show()

def rotacion(boot, w):                                          # fraccion del portafolio que habria que cambiar
    return (np.abs(boot - w.values).sum(axis=1) / 2).mean()
print(f"Rotacion media respecto de la muestra original: tangente = {rotacion(boot_t, w_t):.0%} | PMV = {rotacion(boot_mv, w_mv):.0%}")

**Lectura profesional:** cada caja muestra el rango en que se movió el peso de una acción en las 300 historias. En el tangente hay acciones que van de cero a un tercio del portafolio según la muestra: es la "maximización del error de estimación" de la teoría, vista con datos. El PMV se mueve menos porque no usa $\mu$, el insumo más ruidoso, pero tampoco se queda quieto: con 58 meses estamos estimando 105 varianzas y covarianzas, y eso también es ruido. Por eso en la práctica los portafolios que dependen poco o nada de $\mu$ (mínima varianza, pesos iguales, paridad de riesgo) son competidores serios del "óptimo", y por eso nadie reporta pesos óptimos con dos decimales.

## 5. La defensa más simple: topes por activo

Un tope de 25% por activo (`w_max=0.25`). Pagamos un poco de Sharpe dentro de la muestra a cambio de un portafolio presentable y menos dependiente de dos o tres estimaciones.

In [ ]:
w_t25 = portafolio_tangente(mu, cov, rf, w_max=0.25)
fe25 = frontera_eficiente(mu, cov, n_puntos=30, w_max=0.25)
e25, s25, sh25 = medir(w_t25)
w_eq = pd.Series(1 / len(UNIVERSO), index=UNIVERSO)
e_eq, s_eq, sh_eq = medir(w_eq)

print(f"Tangente libre:        Sharpe = {sh_t:.3f} | activos con peso > 1%: {(w_t > 0.01).sum()} | mayor peso {w_t.max():.0%}")
print(f"Tangente con tope 25%: Sharpe = {sh25:.3f} | activos con peso > 1%: {(w_t25 > 0.01).sum()} | mayor peso {w_t25.max():.0%}")
print(f"Pesos iguales:         Sharpe = {sh_eq:.3f} | activos: {len(UNIVERSO)}")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(fe["riesgo"] * 100, fe["retorno"] * 100, color="#000080", lw=2.5, label="Frontera sin tope")
ax.plot(fe25["riesgo"] * 100, fe25["retorno"] * 100, color="#E36C0A", lw=2.5, ls="--", label="Frontera con tope de 25%")
ax.scatter(s_eq * 100, e_eq * 100, color="gray", s=70, zorder=3, label="Pesos iguales")
ax.set_xlabel("Volatilidad anualizada (%)"); ax.set_ylabel("Retorno medio anualizado (%)")
ax.set_title("Lo que cuesta una restriccion"); ax.grid(alpha=0.3); ax.legend(); plt.tight_layout(); plt.show()

**Para discutir:** toda restricción mueve la frontera hacia adentro: ese es su costo *dentro de la muestra*. Su beneficio aparece fuera de ella. ¿Qué tan lejos queda el humilde portafolio de pesos iguales?

## 6. Reforzamiento rumbo al parcial: un ejercicio por semana

Cada celda rehace con la librería el ejemplo central de una semana y deja una variante para que la resuelvas tú. **Método:** antes de ejecutar la variante, anota qué esperas que pase (¿sube o baja?, ¿mucho o poco?) y luego compara. En el parcial no habrá Python: lo que se evalúa es ese criterio.

In [ ]:
from utils.finanzas import (vpn, tir, capm, wacc, fcff_desde_ebit, fcfe_desde_fcff,
                            valor_terminal, dcf_dos_etapas, per_justificado, valor_por_multiplo_ev,
                            portafolio_dos_activos, peso_minima_varianza)

# Semana 1: invertir 100 y recibir 30, 40 y 60
print(f"S1 | VPN al 10% = {vpn(100, [30, 40, 60], 0.10):.2f} | TIR = {tir(100, [30, 40, 60]):.2%}")
# Variante: sin calcular, el VPN al 12.71% es ...? Y al 15%, positivo o negativo?

# Semana 2: rf 4.5%, beta 1.2, ERP 5.5%, E 700, D 300, Kd 6.5%, t 29.5%
ke = capm(rf=0.045, beta=1.2, erp=0.055)
print(f"S2 | Ke = {ke:.2%} | WACC = {wacc(E=700, D=300, ke=ke, kd=0.065, t=0.295):.2%}")
# Variante: si la empresa se endeuda mas (E 500, D 500) y nada mas cambia, el WACC sube o baja? Que supuesto irreal hay ahi?

# Semana 3: EBIT 100, t 29.5%, depreciacion 30, capex 40, capital de trabajo +10, interes 19.5, deuda neta nueva +10
fcff = fcff_desde_ebit(ebit=100, t=0.295, dep=30, fcinv=40, wcinv=10)
fcfe = fcfe_desde_fcff(fcff, interes=19.5, t=0.295, endeudamiento_neto=10)
print(f"S3 | FCFF = {fcff:.2f} | FCFE = {fcfe:.2f}")
# Variante: con que tasa se descuenta cada uno? Que pasa con el FCFE si el endeudamiento neto fuera -10?

In [ ]:
# Semana 4: FCFF 50.5 creciendo 6, 5.5, 5, 4.5 y 4%, luego g = 4% perpetuo, WACC 9.14%, deuda 300, 100 acciones
proy, f = [], 50.5
for g in [0.06, 0.055, 0.05, 0.045, 0.04]:
    f *= 1 + g; proy.append(f)
res = dcf_dos_etapas(proy, 0.0914, valor_terminal(proy[-1], 0.0914, 0.04))
print(f"S4 | EV = {res['valor']:,.0f} | peso del VT = {res['peso_vt']:.0%} | valor por accion = {(res['valor'] - 300) / 100:.2f}")
# Variante: si g perpetuo sube a 5%, el valor por accion sube poco o mucho? Por que?

# Semana 5: mediana EV/EBITDA 7.5x, EBITDA 130, deuda 300, 100 acciones; payout de capacidad 46.75/56.75
print(f"S5 | Por comparables = {valor_por_multiplo_ev(7.5, 130, 300, 100):.2f} | P/E justificado = {per_justificado(46.75 / 56.75, 0.111, 0.04):.1f}x")
# Variante: un companero aplica 7.5x directamente a la utilidad por accion. Que error conceptual cometio?

# Semana 6: minera (12%, 24%) y electrica (8%, 12%), correlacion 0.25
e, s = portafolio_dos_activos(0.5, 12, 8, 24, 12, 0.25)
print(f"S6 | 50/50: E = {e:.1f}% y sigma = {s:.1f}% | peso de minima varianza en la minera = {peso_minima_varianza(24, 12, 0.25):.1%}")
# Variante: con correlacion 1 la volatilidad del 50/50 seria ...? Y por que nunca puede ser mayor que eso?

Salidas esperadas: S1: 5.41 y 12.71%. S2: 11.10% y 9.14%. S3: 50.50 y 46.75. S4: EV de 1,069 con el VT pesando 79% y 7.69 por acción. S5: 6.75 y 12.1x. S6: 10.0%, 14.7% y 12.5%.

Pistas para las variantes: S1, el VPN a la TIR es cero por definición, y al 15% es negativo. S2, el WACC baja, pero el supuesto irreal es que $K_e$ y $K_d$ no suban con el apalancamiento (semana 2: reapalancar el beta). S3, FCFF con WACC y FCFE con $K_e$; amortizar deuda reduce el FCFE. S4, mucho: el valor terminal pesa casi todo y $g$ está en el denominador. S5, aplicó un múltiplo de firma a una métrica del accionista y además se saltó el puente. S6, 18%, el promedio ponderado: es el techo porque $\rho$ no pasa de 1.

## 7. Cierre

Desde hoy quedan en la librería: `min_varianza()`, `portafolio_tangente()` y `frontera_eficiente()`, las tres con `cortos` y `w_max` como opciones.

**Tarea** (`clase07_tarea.ipynb`): la frontera eficiente de tu universo de seis activos. Es corta a propósito. Entrega hasta el lunes, vía commit.

**Examen parcial: miércoles 21/10**, semanas 1 a 7. Repasa con los notebooks de apuntes (listas de verificación y respuestas a los ítems) y con el mock de la PC1 en `mocks/`.

**Después del parcial (semana 9):** del tangente al mercado: CAPM, la línea del mercado de valores y los modelos multifactoriales.